# OSCC Histopathology Classification

Computer Vision Final Project — single-notebook implementation of the pipeline described in `Project_steps_v2.md`:

1. Setup
2. Config
3. Data acquisition
4. EDA
5. Patient-level split
6. Stain normalization (with on-disk cache)
7. Dataset and transforms
8. Model factory (AlexNet / ResNet50 / EfficientNet-B0 / ViT-Tiny)
9. Training loop (weighted CE, AdamW, cosine LR, early stopping on val AUC)
10. Evaluation (accuracy / precision / recall / F1 / AUC, confusion matrix, ROC)
11. Grad-CAM visualization
12. Experiment runner over the experiment matrix


## 1. Setup

Mount Google Drive, install missing dependencies, import everything.

In [7]:
# Mount Google Drive (Colab only). Comment out if running locally.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("In Colab:", IN_COLAB)


Mounted at /content/drive
In Colab: True


In [8]:
# Install dependencies that are not part of the default Colab image.
%pip install -q timm torchstain grad-cam


In [9]:
import os
import json
import random
import zipfile
import hashlib
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

import timm

from sklearn.model_selection import GroupShuffleSplit, StratifiedShuffleSplit
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch:", torch.__version__, "| Device:", device)


Torch: 2.10.0+cu128 | Device: cuda


## 2. Config

Single source of truth for all hyper-parameters and paths.
Override entries via `run_experiment({...})` later in the notebook.

In [10]:
CONFIG = {
    "data_root":     "/content/drive/MyDrive/CV_Final/OSCC",
    "magnification": "400x",                  # "100x" | "400x"
    "stain_norm":    "reinhard",              # "none" | "reinhard" | "macenko"
    "backbone":      "resnet50",              # "alexnet" | "resnet50" | "efficientnet_b0" | "vit_tiny_patch16_224"
    "strategy":      "finetune",              # "scratch" | "finetune" | "freeze"
    "img_size":      224,
    "batch_size":    32,
    "epochs":        40,
    "lr":            1e-4,
    "weight_decay":  1e-4,
    "patience":      7,
    "seed":          42,
    "out_dir":       "/content/drive/MyDrive/CV_Final/results",
    "num_workers":   0,                       # 0 avoids the noisy "can only test a child process" warnings on Colab/Jupyter
}

# If running locally instead of Colab, fall back to the current working directory.
if not IN_COLAB:
    CONFIG["data_root"] = "./data/OSCC"
    CONFIG["out_dir"]   = "./results"

Path(CONFIG["out_dir"]).mkdir(parents=True, exist_ok=True)


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(CONFIG["seed"])


## 3. Data Acquisition

Mendeley dataset `ftmp4cvtmb` (Rahman et al., 2019). The first cell downloads and unzips
the archive into `CONFIG["data_root"]` if it is not already there. The download link can
break occasionally; if so, download manually from
<https://data.mendeley.com/datasets/ftmp4cvtmb/1> and put the unzipped folder at the same path.

In [11]:
DATA_URL = "https://prod-dcd-datasets-cache-zipfiles.s3.eu-west-1.amazonaws.com/ftmp4cvtmb-1.zip"
MIN_VALID_ZIP_BYTES = 100 * 1024 * 1024   # >= 100 MB → assume the archive is intact (real one is ~2.9 GB)

# Force-remount Drive on Colab so files uploaded during the current session become visible.
if IN_COLAB:
    from google.colab import drive as _drive
    try:
        _drive.mount('/content/drive', force_remount=True)
    except Exception as e:
        print(f"Drive remount skipped: {e}")

data_root = Path(CONFIG["data_root"])
data_root.mkdir(parents=True, exist_ok=True)


def _find_existing_zip(folder: Path) -> Path | None:
    """Return the largest .zip in `folder` whose name contains 'ftmp4cvtmb' (handles
    auto-renamed duplicates like 'ftmp4cvtmb-1 (1).zip').
    """
    candidates = [p for p in folder.glob("*.zip") if "ftmp4cvtmb" in p.name.lower()]
    if not candidates:
        return None
    return max(candidates, key=lambda p: p.stat().st_size)


existing_images = list(data_root.rglob("*.jpg")) + list(data_root.rglob("*.png"))
zip_path = _find_existing_zip(data_root) or (data_root / "ftmp4cvtmb-1.zip")
zip_size = zip_path.stat().st_size if zip_path.exists() else 0
zip_ok = zip_size >= MIN_VALID_ZIP_BYTES
print(f"Found {len(existing_images)} extracted images; "
      f"zip_path={zip_path.name!r} size={zip_size/1e6:.1f} MB (valid={zip_ok})")

if len(existing_images) >= 100:
    print(f"Dataset already extracted at {data_root}.")
elif zip_ok:
    print(f"Extracting existing zip {zip_path} ...")
    try:
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(data_root)
        print("Extraction done.")
    except zipfile.BadZipFile:
        print(f"ERROR: {zip_path} is not a valid zip even though it is {zip_size/1e6:.1f} MB. "
              "Delete it and re-upload from https://data.mendeley.com/datasets/ftmp4cvtmb/1.")
else:
    if zip_path.exists():
        print(f"ABORTING: {zip_path} exists but is only {zip_size/1e6:.1f} MB "
              "(expected ~2900 MB). It may be a partial upload — wait for the upload to finish "
              "or delete it and re-upload. Not running wget to avoid overwriting your file.")
    else:
        target = data_root / "ftmp4cvtmb-1.zip"
        print(f"Downloading dataset to {target} ...")
        !wget -q -O "{target}" "{DATA_URL}"
        downloaded = target.stat().st_size if target.exists() else 0
        if downloaded < MIN_VALID_ZIP_BYTES:
            print(f"Download failed (got {downloaded/1e6:.1f} MB). "
                  "Please download manually from "
                  "https://data.mendeley.com/datasets/ftmp4cvtmb/1 and upload the zip to "
                  f"{target}, then re-run this cell.")
            target.unlink(missing_ok=True)
        else:
            print(f"Extracting {target} ...")
            with zipfile.ZipFile(target, "r") as zf:
                zf.extractall(data_root)
            print("Extraction done.")


Mounted at /content/drive
Found 1224 extracted images; zip_path='ftmp4cvtmb-1.zip' size=3098.6 MB (valid=True)
Dataset already extracted at /content/drive/MyDrive/CV_Final/OSCC.


In [12]:
def parse_dataset(root: Path) -> pd.DataFrame:
    """Walk the dataset folder and infer (label, magnification, patient_id) per image.

    Uses ONLY the path components BELOW `root` (so the dataset's parent folder name,
    which often contains 'OSCC', does not contaminate label detection).
    Skips stain-normalization caches (any directory named `normalized_*`).
    """
    rows, skipped_cache = [], 0
    root = Path(root)
    for path in root.rglob("*"):
        if path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".tif", ".tiff"}:
            continue
        try:
            rel_parts = [p.lower() for p in path.relative_to(root).parts]
        except ValueError:
            continue

        # Skip the on-disk stain-normalization cache.
        if any(p.startswith("normalized_") for p in rel_parts):
            skipped_cache += 1
            continue

        rel_str = "/".join(rel_parts)

        # Label: cancer keywords first; "normal" must be a folder, not a substring of
        # something else. (We already excluded `normalized_*` above.)
        if "oscc" in rel_str or "cancer" in rel_str or "malignant" in rel_str:
            label = 1
        elif "normal" in rel_str:
            label = 0
        else:
            continue

        if "100" in rel_str or "first set" in rel_str or "first_set" in rel_str:
            mag = "100x"
        elif "400" in rel_str or "second set" in rel_str or "second_set" in rel_str:
            mag = "400x"
        else:
            mag = "unknown"

        # Patient ID is not encoded in the public Mendeley filenames in a reliable way.
        rows.append({
            "path": str(path),
            "label": label,
            "magnification": mag,
            "patient_id": None,
        })

    if skipped_cache:
        print(f"(skipped {skipped_cache} images under normalized_* cache directories)")
    return pd.DataFrame(rows)


df = parse_dataset(data_root)
print(f"Parsed {len(df)} images.")
counts = df.groupby(["magnification", "label"]).size().unstack(fill_value=0)
print(counts)

# Sanity checks.
if df.empty:
    raise RuntimeError("parse_dataset returned an empty DataFrame. "
                       "Check the dataset folder structure with: "
                       f"!find {data_root} -type d -not -name 'normalized_*' | head -30")
if 0 not in df["label"].unique() or 1 not in df["label"].unique():
    raise RuntimeError(f"parse_dataset only found classes {sorted(df['label'].unique())}. "
                       "Inspect the folder names above; the parser expects 'Normal' / 'OSCC' subfolders.")
if "unknown" in counts.index and int(counts.loc["unknown"].sum()) > 0:
    n_unknown = int(counts.loc["unknown"].sum())
    print(f"\nWARNING: {n_unknown} images could not be assigned to 100x/400x. "
          "Sample paths:")
    print(df[df["magnification"] == "unknown"]["path"].head().tolist())

df.head()


Parsed 1224 images.
label            0    1
magnification          
100x            90  440
400x           200  494


,path,label,magnification,patient_id
0,/content/drive/MyDrive/CV_Final/OSCC/Second Se...,1,400x,None
1,/content/drive/MyDrive/CV_Final/OSCC/Second Se...,1,400x,None
2,/content/drive/MyDrive/CV_Final/OSCC/Second Se...,1,400x,None
3,/content/drive/MyDrive/CV_Final/OSCC/Second Se...,1,400x,None
4,/content/drive/MyDrive/CV_Final/OSCC/Second Se...,1,400x,None


## 4. EDA

Quick sanity check: class counts and a few sample patches.

In [ ]:
def show_sample_grid(df: pd.DataFrame, n_per_cell: int = 2):
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    for r, mag in enumerate(["100x", "400x"]):
        for c, label in enumerate([0, 1]):
            sub = df[(df["magnification"] == mag) & (df["label"] == label)]
            for k in range(n_per_cell):
                ax = axes[r, c * n_per_cell + k]
                if len(sub) == 0:
                    ax.axis("off")
                    continue
                row = sub.sample(1, random_state=42 + k).iloc[0]
                img = Image.open(row["path"]).convert("RGB")
                ax.imshow(img)
                ax.set_title(f"{mag} | {'OSCC' if label==1 else 'Normal'}")
                ax.axis("off")
    plt.tight_layout()
    plt.show()


show_sample_grid(df)


## 5. Patient-level Split

70 / 15 / 15. If patient IDs are unavailable (the public Mendeley filenames do not encode them
reliably), we fall back to a stratified random split and print a warning.

In [ ]:
def split_dataset(df: pd.DataFrame, val_size=0.15, test_size=0.15, seed=42):
    """Patient-level split when patient IDs exist; otherwise a stratified split that
    preserves the joint (label, magnification) distribution so each (split, magnification)
    bucket contains both classes."""
    df = df.reset_index(drop=True)
    has_pids = df["patient_id"].notna().any()

    if has_pids:
        groups = df["patient_id"].fillna(df.index.astype(str))
        gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
        rest_idx, test_idx = next(gss.split(df, df["label"], groups=groups))
        rest_df = df.iloc[rest_idx].reset_index(drop=True)
        rest_groups = rest_df["patient_id"].fillna(rest_df.index.astype(str))
        gss2 = GroupShuffleSplit(n_splits=1, test_size=val_size / (1 - test_size), random_state=seed)
        train_idx, val_idx = next(gss2.split(rest_df, rest_df["label"], groups=rest_groups))
        train_df = rest_df.iloc[train_idx].reset_index(drop=True)
        val_df = rest_df.iloc[val_idx].reset_index(drop=True)
        test_df = df.iloc[test_idx].reset_index(drop=True)
    else:
        warnings.warn("Patient IDs not available; falling back to stratified random split "
                      "on (label, magnification). Document this as a limitation in the report.")
        # Joint stratification key keeps both classes present in every (split, magnification) bucket.
        strat = df["magnification"].astype(str) + "_" + df["label"].astype(str)
        sss1 = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
        rest_idx, test_idx = next(sss1.split(df, strat))
        rest_df = df.iloc[rest_idx].reset_index(drop=True)
        rest_strat = rest_df["magnification"].astype(str) + "_" + rest_df["label"].astype(str)
        sss2 = StratifiedShuffleSplit(n_splits=1, test_size=val_size / (1 - test_size), random_state=seed)
        train_idx, val_idx = next(sss2.split(rest_df, rest_strat))
        train_df = rest_df.iloc[train_idx].reset_index(drop=True)
        val_df = rest_df.iloc[val_idx].reset_index(drop=True)
        test_df = df.iloc[test_idx].reset_index(drop=True)
    return train_df, val_df, test_df


train_df, val_df, test_df = split_dataset(df, seed=CONFIG["seed"])
print("train:", len(train_df), "val:", len(val_df), "test:", len(test_df))
print("\nPer-(split, magnification, label) counts:")
for name, sub in [("train", train_df), ("val", val_df), ("test", test_df)]:
    pivot = sub.groupby(["magnification", "label"]).size().unstack(fill_value=0)
    print(f"\n[{name}]")
    print(pivot)


## 6. Stain Normalization

`StainNormalizer` wraps `torchstain` and supports `none | reinhard | macenko`.
The first OSCC training image is used as the reference. Normalized images are cached on disk
under `data_root/normalized_<method>/` so subsequent runs are cheap.

In [ ]:
class StainNormalizer:
    def __init__(self, method: str, reference_path: str = None):
        self.method = method
        if method == "none":
            return
        import torchstain
        self._torchstain = torchstain
        self._T_pre = transforms.Compose([
            transforms.ToTensor(),
            transforms.Lambda(lambda x: x * 255.0),
        ])
        if method == "reinhard":
            self.normalizer = torchstain.normalizers.ReinhardNormalizer(backend="torch")
        elif method == "macenko":
            self.normalizer = torchstain.normalizers.MacenkoNormalizer(backend="torch")
        else:
            raise ValueError(f"Unknown stain method: {method}")
        ref = Image.open(reference_path).convert("RGB")
        self.normalizer.fit(self._T_pre(ref))

    def __call__(self, img: Image.Image) -> Image.Image:
        if self.method == "none":
            return img
        try:
            t = self._T_pre(img)
            if self.method == "reinhard":
                norm = self.normalizer.normalize(I=t)
            else:
                out = self.normalizer.normalize(I=t, stains=False)
                norm = out[0] if isinstance(out, tuple) else out
            arr = norm.numpy() if hasattr(norm, "numpy") else np.array(norm)
            arr = np.clip(arr, 0, 255).astype(np.uint8)
            if arr.shape[0] == 3 and arr.ndim == 3 and arr.shape[-1] != 3:
                arr = np.transpose(arr, (1, 2, 0))
            return Image.fromarray(arr)
        except Exception as e:
            warnings.warn(f"Stain norm failed: {e}; returning original.")
            return img


def stain_normalize_cache(df: pd.DataFrame, method: str, cache_root: str, reference_path: str) -> pd.DataFrame:
    """Run stain normalization once and cache outputs to disk; return df with new `norm_path` column."""
    df = df.copy()
    if method == "none":
        df["norm_path"] = df["path"]
        return df

    cache_dir = Path(cache_root) / f"normalized_{method}"
    cache_dir.mkdir(parents=True, exist_ok=True)

    normalizer = StainNormalizer(method, reference_path)
    out_paths = []
    for src in tqdm(df["path"], desc=f"Stain norm ({method})"):
        h = hashlib.md5(src.encode()).hexdigest()[:10]
        out = cache_dir / f"{h}_{Path(src).name}"
        if not out.exists():
            try:
                img = Image.open(src).convert("RGB")
                normalizer(img).save(out)
            except Exception as e:
                warnings.warn(f"Failed on {src}: {e}")
                Image.open(src).convert("RGB").save(out)
        out_paths.append(str(out))
    df["norm_path"] = out_paths
    return df


## 7. Dataset and Transforms

In [ ]:
class OSCCDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform, path_col: str = "norm_path"):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.path_col = path_col if path_col in df.columns else "path"

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row[self.path_col]).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        return img, int(row["label"])


def build_transforms(img_size: int = 224, train: bool = True):
    norm = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                std=[0.229, 0.224, 0.225])
    if train:
        return transforms.Compose([
            transforms.RandomResizedCrop(img_size, scale=(0.8, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomChoice([
                transforms.RandomRotation((0, 0)),
                transforms.RandomRotation((90, 90)),
                transforms.RandomRotation((180, 180)),
                transforms.RandomRotation((270, 270)),
            ]),
            transforms.ColorJitter(brightness=0.1, contrast=0.1),
            transforms.ToTensor(),
            norm,
        ])
    return transforms.Compose([
        transforms.Resize(int(img_size * 1.14)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        norm,
    ])


## 8. Model Factory

`build_model(backbone, strategy)` returns an `nn.Module` with the original head replaced by:

```
GAP -> Linear(feat, 256) -> ReLU -> Dropout(0.5) -> Linear(256, 2)
```

Three strategies: `scratch` (no pretraining), `finetune` (full network from ImageNet),
`freeze` (linear probing).

In [ ]:
class ANNHead(nn.Module):
    def __init__(self, in_features: int, num_classes: int = 2,
                 hidden: int = 256, dropout: float = 0.5):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(in_features, hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_classes),
        )

    def forward(self, x):
        return self.head(x)


def _alexnet(pretrained: bool, num_classes: int):
    weights = models.AlexNet_Weights.IMAGENET1K_V1 if pretrained else None
    m = models.alexnet(weights=weights)
    in_feat = m.classifier[6].in_features
    m.classifier[6] = ANNHead(in_feat, num_classes)
    head_params = list(m.classifier[6].parameters())
    return m, head_params


def _resnet50(pretrained: bool, num_classes: int):
    weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
    m = models.resnet50(weights=weights)
    in_feat = m.fc.in_features
    m.fc = ANNHead(in_feat, num_classes)
    head_params = list(m.fc.parameters())
    return m, head_params


def _timm_with_head(name: str, pretrained: bool, num_classes: int):
    backbone = timm.create_model(name, pretrained=pretrained, num_classes=0)
    head = ANNHead(backbone.num_features, num_classes)
    m = nn.Sequential(backbone, head)
    head_params = list(head.parameters())
    return m, head_params


def build_model(backbone: str, strategy: str, num_classes: int = 2) -> nn.Module:
    pretrained = strategy in {"finetune", "freeze"}
    if backbone == "alexnet":
        model, head_params = _alexnet(pretrained, num_classes)
    elif backbone == "resnet50":
        model, head_params = _resnet50(pretrained, num_classes)
    elif backbone == "efficientnet_b0":
        model, head_params = _timm_with_head("efficientnet_b0", pretrained, num_classes)
    elif backbone == "vit_tiny_patch16_224":
        model, head_params = _timm_with_head("vit_tiny_patch16_224", pretrained, num_classes)
    else:
        raise ValueError(f"Unknown backbone: {backbone}")

    if strategy == "freeze":
        for p in model.parameters():
            p.requires_grad = False
        for p in head_params:
            p.requires_grad = True
    return model


## 9. Training Loop

Weighted cross-entropy, AdamW, cosine LR, early stopping on val AUC.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, n = 0.0, 0
    for x, y in tqdm(loader, desc="train", leave=False):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        n += x.size(0)
    return total_loss / max(n, 1)


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, n = 0.0, 0
    all_probs, all_labels = [], []
    for x, y in tqdm(loader, desc="eval", leave=False):
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        total_loss += loss.item() * x.size(0)
        n += x.size(0)
        probs = F.softmax(logits, dim=1)[:, 1]
        all_probs.append(probs.cpu().numpy())
        all_labels.append(y.cpu().numpy())
    probs = np.concatenate(all_probs) if all_probs else np.array([])
    labels = np.concatenate(all_labels) if all_labels else np.array([])
    try:
        auc = roc_auc_score(labels, probs)
    except ValueError:
        auc = float("nan")
    return total_loss / max(n, 1), auc, probs, labels


class EarlyStopper:
    def __init__(self, patience: int = 7, mode: str = "max"):
        self.patience = patience
        self.mode = mode
        self.best = -np.inf if mode == "max" else np.inf
        self.counter = 0
        self.should_stop = False

    def step(self, value: float) -> bool:
        improved = (value > self.best) if self.mode == "max" else (value < self.best)
        if improved:
            self.best = value
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
        return improved


In [ ]:
def make_run_name(cfg: dict) -> str:
    return f"{cfg['magnification']}_{cfg['stain_norm']}_{cfg['backbone']}_{cfg['strategy']}"


def train_model(cfg: dict, train_df: pd.DataFrame, val_df: pd.DataFrame):
    set_seed(cfg["seed"])
    out_dir = Path(cfg["out_dir"])
    out_dir.mkdir(parents=True, exist_ok=True)
    run_name = make_run_name(cfg)

    # Sanity-check the split: warn loudly if either side has a single class.
    train_classes = sorted(set(train_df["label"]))
    val_classes   = sorted(set(val_df["label"]))
    print(f"[{run_name}] train n={len(train_df)} classes={train_classes} "
          f"| val n={len(val_df)} classes={val_classes}")
    if len(val_classes) < 2:
        warnings.warn(f"Validation set contains only class(es) {val_classes}; "
                      f"AUC will be NaN. Re-check split_dataset / magnification filtering.")

    train_ds = OSCCDataset(train_df, build_transforms(cfg["img_size"], train=True))
    val_ds   = OSCCDataset(val_df,   build_transforms(cfg["img_size"], train=False))
    train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True,
                              num_workers=cfg["num_workers"], pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=cfg["batch_size"], shuffle=False,
                              num_workers=cfg["num_workers"], pin_memory=True)

    counts = Counter(train_df["label"].tolist())
    inv = np.array([1.0 / counts.get(i, 1) for i in range(2)], dtype=np.float32)
    inv = inv / inv.sum() * 2.0  # mean weight ~ 1
    weights = torch.tensor(inv, device=device)
    criterion = nn.CrossEntropyLoss(weight=weights)

    model = build_model(cfg["backbone"], cfg["strategy"]).to(device)
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable, lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["epochs"])
    stopper = EarlyStopper(patience=cfg["patience"], mode="max")

    history, best_path = [], out_dir / f"{run_name}.pt"
    # Always save an initial checkpoint so downstream evaluate_model never sees a missing file.
    torch.save(model.state_dict(), best_path)
    best_val_loss = float("inf")
    for epoch in range(1, cfg["epochs"] + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_auc, _, _ = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        # Pick the model-selection metric: AUC if defined, otherwise -val_loss as a fallback.
        sel_metric = val_auc if not np.isnan(val_auc) else -val_loss
        improved = stopper.step(sel_metric)

        # Independently track best val_loss so we always save *something* sensible.
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            if np.isnan(val_auc):
                torch.save(model.state_dict(), best_path)

        history.append({"epoch": epoch, "train_loss": train_loss,
                        "val_loss": val_loss, "val_auc": val_auc})
        marker = " *" if improved else ""
        print(f"[{run_name}] epoch {epoch:03d} "
              f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_auc={val_auc:.4f}{marker}")
        if improved and not np.isnan(val_auc):
            torch.save(model.state_dict(), best_path)
        if stopper.should_stop:
            print(f"Early stop at epoch {epoch} (best metric={stopper.best:.4f})")
            break

    pd.DataFrame(history).to_csv(out_dir / f"{run_name}_history.csv", index=False)
    return best_path, stopper.best


## 10. Evaluation

Load the best checkpoint and report all metrics + confusion matrix + ROC.

In [ ]:
def evaluate_model(cfg: dict, test_df: pd.DataFrame, ckpt_path: Path):
    model = build_model(cfg["backbone"], cfg["strategy"]).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))

    test_ds = OSCCDataset(test_df, build_transforms(cfg["img_size"], train=False))
    test_loader = DataLoader(test_ds, batch_size=cfg["batch_size"], shuffle=False,
                             num_workers=cfg["num_workers"], pin_memory=True)

    _, auc, probs, labels = evaluate(model, test_loader, nn.CrossEntropyLoss(), device)
    preds = (probs >= 0.5).astype(int)
    metrics = {
        "accuracy":  accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall":    recall_score(labels, preds, zero_division=0),
        "f1":        f1_score(labels, preds, zero_division=0),
        "auc":       float(auc),
    }
    print(json.dumps(metrics, indent=2))

    out_dir = Path(cfg["out_dir"])
    run_name = make_run_name(cfg)

    cm = confusion_matrix(labels, preds, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["Normal", "OSCC"]); ax.set_yticklabels(["Normal", "OSCC"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title("Confusion Matrix")
    plt.tight_layout()
    plt.savefig(out_dir / f"{run_name}_cm.png", dpi=150)
    plt.show()

    fpr, tpr, _ = roc_curve(labels, probs)
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot(fpr, tpr, label=f"AUC={auc:.3f}")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Curve"); ax.legend()
    plt.tight_layout()
    plt.savefig(out_dir / f"{run_name}_roc.png", dpi=150)
    plt.show()

    return metrics, probs, labels, preds


## 11. Grad-CAM Visualization

Pick one TP, TN, FP, FN sample from the test set and overlay the activation heatmap on the
input image. ViT uses the last attention block's `norm1` as the target layer.

In [ ]:
def gradcam_visualization(cfg: dict, test_df: pd.DataFrame, ckpt_path: Path,
                          probs: np.ndarray, labels: np.ndarray, preds: np.ndarray):
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image

    model = build_model(cfg["backbone"], cfg["strategy"]).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()

    if cfg["backbone"] == "resnet50":
        target_layers = [model.layer4[-1]]
    elif cfg["backbone"] == "alexnet":
        target_layers = [model.features[-1]]
    elif cfg["backbone"] == "efficientnet_b0":
        target_layers = [model[0].conv_head]
    elif cfg["backbone"] == "vit_tiny_patch16_224":
        target_layers = [model[0].blocks[-1].norm1]
    else:
        raise ValueError(f"Grad-CAM not configured for {cfg['backbone']}")

    df = test_df.reset_index(drop=True).copy()
    df["pred"] = preds
    df["prob"] = probs

    def pick(true_label, pred_label):
        sub = df[(df["label"] == true_label) & (df["pred"] == pred_label)]
        return sub.iloc[0] if len(sub) else None

    samples = {"TP": pick(1, 1), "TN": pick(0, 0), "FP": pick(0, 1), "FN": pick(1, 0)}

    cam = GradCAM(model=model, target_layers=target_layers)
    eval_tf = build_transforms(cfg["img_size"], train=False)

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    for i, (name, row) in enumerate(samples.items()):
        if row is None:
            for r in range(2):
                axes[r, i].axis("off")
                axes[r, i].set_title(f"{name}: n/a")
            continue
        path = row.get("norm_path", row["path"])
        img = Image.open(path).convert("RGB").resize((cfg["img_size"], cfg["img_size"]))
        rgb = np.array(img).astype(np.float32) / 255.0
        x = eval_tf(Image.fromarray((rgb * 255).astype(np.uint8))).unsqueeze(0).to(device)
        gray_cam = cam(input_tensor=x, targets=None)[0]
        overlay = show_cam_on_image(rgb, gray_cam, use_rgb=True)
        axes[0, i].imshow(rgb);     axes[0, i].axis("off")
        axes[0, i].set_title(f"{name} (p={row['prob']:.2f})")
        axes[1, i].imshow(overlay); axes[1, i].axis("off")
    plt.tight_layout()
    out_path = Path(cfg["out_dir"]) / f"{make_run_name(cfg)}_gradcam.png"
    plt.savefig(out_path, dpi=150)
    plt.show()
    print(f"Saved {out_path}")


## 12. Experiment Runner

`run_experiment(overrides, ...)` filters the splits to the chosen magnification, applies stain
normalization caching, trains, evaluates, and appends one row to `results.csv`.

The three experiment matrices from `Project_steps_v2.md` are pre-defined below as Python lists
and **commented out**. Uncomment the block(s) you want to run.

In [ ]:
RESULTS_CSV = Path(CONFIG["out_dir"]) / "results.csv"


def run_experiment(overrides: dict,
                   train_df_all: pd.DataFrame,
                   val_df_all:   pd.DataFrame,
                   test_df_all:  pd.DataFrame,
                   results_csv:  Path = RESULTS_CSV,
                   do_gradcam:   bool = False):
    cfg = {**CONFIG, **overrides}

    tr = train_df_all[train_df_all["magnification"] == cfg["magnification"]].reset_index(drop=True)
    va = val_df_all  [val_df_all  ["magnification"] == cfg["magnification"]].reset_index(drop=True)
    te = test_df_all [test_df_all ["magnification"] == cfg["magnification"]].reset_index(drop=True)
    if len(tr) == 0:
        raise RuntimeError(f"No training samples for magnification {cfg['magnification']}.")

    ref_path = tr[tr["label"] == 1]["path"].iloc[0]
    tr2 = stain_normalize_cache(tr, cfg["stain_norm"], cfg["data_root"], ref_path)
    va2 = stain_normalize_cache(va, cfg["stain_norm"], cfg["data_root"], ref_path)
    te2 = stain_normalize_cache(te, cfg["stain_norm"], cfg["data_root"], ref_path)

    ckpt, best_val_auc = train_model(cfg, tr2, va2)
    metrics, probs, labels, preds = evaluate_model(cfg, te2, ckpt)

    if do_gradcam:
        gradcam_visualization(cfg, te2, ckpt, probs, labels, preds)

    row = {
        "magnification": cfg["magnification"],
        "stain_norm":    cfg["stain_norm"],
        "backbone":      cfg["backbone"],
        "strategy":      cfg["strategy"],
        "best_val_auc":  best_val_auc,
        **metrics,
    }
    df_out = pd.DataFrame([row])
    if results_csv.exists():
        df_out = pd.concat([pd.read_csv(results_csv), df_out], ignore_index=True)
    df_out.to_csv(results_csv, index=False)
    print(f"Logged to {results_csv}")
    return row, ckpt, te2, probs, labels, preds


# ----------------------------------------------------------------
# Demo: a single Run-All friendly experiment using CONFIG defaults.
# Uncomment to execute.
# ----------------------------------------------------------------
# row, ckpt, te2, probs, labels, preds = run_experiment(
#     overrides={}, train_df_all=train_df, val_df_all=val_df, test_df_all=test_df,
#     do_gradcam=True,
# )

# ----------------------------------------------------------------
# Experiment 1 - Stain normalization ablation (ResNet50 + finetune)
# ----------------------------------------------------------------
# EXP1 = [
#     {"magnification": mag, "stain_norm": s, "backbone": "resnet50", "strategy": "finetune"}
#     for mag in ["100x", "400x"]
#     for s   in ["none", "reinhard", "macenko"]
# ]
# for ov in EXP1:
#     run_experiment(ov, train_df, val_df, test_df)

# ----------------------------------------------------------------
# Experiment 2 - Backbone x Strategy on the 400x set
# (use the best stain_norm from EXP1)
# ----------------------------------------------------------------
# EXP2 = []
# for backbone, strategies in [
#     ("alexnet",                ["scratch", "finetune", "freeze"]),
#     ("resnet50",               ["scratch", "finetune", "freeze"]),
#     ("efficientnet_b0",        ["finetune", "freeze"]),
#     ("vit_tiny_patch16_224",   ["finetune", "freeze"]),
# ]:
#     for s in strategies:
#         EXP2.append({"magnification": "400x", "stain_norm": "reinhard",
#                      "backbone": backbone, "strategy": s})
# for ov in EXP2:
#     run_experiment(ov, train_df, val_df, test_df)

# ----------------------------------------------------------------
# Experiment 3 - Magnification comparison (use the best model from EXP2)
# ----------------------------------------------------------------
# EXP3 = [
#     {"magnification": "100x", "stain_norm": "reinhard", "backbone": "resnet50", "strategy": "finetune"},
#     {"magnification": "400x", "stain_norm": "reinhard", "backbone": "resnet50", "strategy": "finetune"},
# ]
# for ov in EXP3:
#     run_experiment(ov, train_df, val_df, test_df)
